# 19. FCN 핵심 아이디어

이 노트북은 `18_Semantic_Segmentation_개념.ipynb` 다음 단계로, FCN(Fully Convolutional Network)이 CNN 분류 모델을 어떻게 segmentation 모델로 바꾸었는지 이해하는 것이 목표입니다.

CNN 분류 모델은 이미지 전체를 하나의 클래스 확률로 요약합니다. 하지만 semantic segmentation은 픽셀마다 클래스를 예측해야 합니다. FCN의 핵심은 이 차이를 해결하기 위해 **fully connected layer를 convolution으로 바꾸고, coarse feature map을 다시 키워 dense prediction을 수행**하는 것입니다.

이번 노트북의 목표는 다음과 같습니다.

- 분류 CNN이 왜 픽셀 단위 예측에 그대로 쓰기 어려운지 이해합니다.
- fully connected layer와 convolution layer의 역할 차이를 정리합니다.
- FCN의 `convolutionalization`, `upsampling`, `skip connection` 아이디어를 이해합니다.
- U-Net으로 넘어가기 전에 encoder-decoder 관점을 준비합니다.


## 19-1. 준비

이번 노트북은 실제 FCN을 학습하지 않습니다. 대신 feature map의 크기 변화를 단순화해서 FCN이 어떤 문제를 해결했는지 확인합니다.


In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False


## 19-2. 분류 CNN의 마지막은 공간 정보를 압축한다

일반적인 CNN 분류 모델은 convolution과 pooling을 반복하면서 feature map의 공간 크기를 줄입니다. 마지막에는 global average pooling 또는 fully connected layer를 거쳐 이미지 전체를 하나의 벡터로 요약합니다.

이 구조는 classification에는 적합합니다. 하지만 segmentation에서는 문제가 됩니다.

- classification: 이미지 전체의 대표 클래스만 알면 됨
- segmentation: 각 픽셀이 어떤 클래스인지 알아야 함

즉, segmentation에서는 마지막까지 공간 위치 정보를 어느 정도 유지해야 합니다.


In [ ]:
stages = ['input', 'conv1', 'pool1', 'conv2', 'pool2', 'conv3', 'classifier']
sizes = [224, 112, 56, 28, 14, 7, 1]

plt.figure(figsize=(8, 4))
plt.plot(stages, sizes, marker='o')
plt.title('분류 CNN에서 공간 해상도 감소')
plt.ylabel('feature map size')
plt.grid(True, axis='y')
plt.show()


## 19-3. Fully connected layer를 convolution으로 바꾸기

FCN의 첫 번째 핵심은 분류 CNN의 fully connected layer를 convolution layer로 바꾸는 것입니다.

Fully connected layer는 고정된 길이의 벡터를 입력으로 받기 때문에 입력 이미지 크기가 고정되는 경향이 있습니다. 반면 convolution layer는 feature map 위를 같은 필터로 훑으므로 공간 구조를 유지한 채 점수를 만들 수 있습니다.

이렇게 바꾸면 모델은 이미지 전체 클래스 하나가 아니라, 낮은 해상도의 class score map을 출력할 수 있습니다.


In [ ]:
num_classes = 3
feature_h, feature_w = 7, 7

classification_output = np.zeros((num_classes,))
fcn_score_map = np.zeros((num_classes, feature_h, feature_w))

print('분류 CNN 출력 shape:', classification_output.shape)
print('FCN score map 출력 shape:', fcn_score_map.shape)


위 출력의 차이가 중요합니다. 분류 CNN은 클래스별 점수 벡터 하나를 만들지만, FCN은 각 위치마다 클래스별 점수를 가진 score map을 만듭니다.


## 19-4. 낮은 해상도 score map을 다시 키우기

CNN backbone을 지나면 feature map의 해상도는 입력 이미지보다 작아집니다. 예를 들어 입력이 `224 x 224`인데 마지막 score map이 `7 x 7`이면, 이 상태로는 픽셀 단위 예측 결과라고 보기 어렵습니다.

FCN은 score map을 입력 이미지 크기까지 키우는 upsampling을 사용합니다. 대표적으로 transposed convolution 또는 bilinear interpolation을 사용할 수 있습니다.


In [ ]:
coarse_map = np.array([
    [0, 0, 1, 1],
    [0, 1, 1, 1],
    [2, 2, 1, 1],
    [2, 2, 2, 1],
])

upsampled_map = np.repeat(np.repeat(coarse_map, 4, axis=0), 4, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(coarse_map, cmap='tab10', vmin=0, vmax=9)
axes[0].set_title('Coarse score map')
axes[1].imshow(upsampled_map, cmap='tab10', vmin=0, vmax=9)
axes[1].set_title('Upsampled prediction')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()


단순 반복 upsampling은 개념 이해용 예시입니다. 실제 모델에서는 더 부드럽고 학습 가능한 방식으로 해상도를 복원할 수 있습니다. 핵심은 **작아진 feature map을 다시 원래 이미지 크기에 맞춘다**는 점입니다.


## 19-5. Skip connection이 필요한 이유

깊은 feature map은 의미 정보가 풍부합니다. 예를 들어 `이 영역은 자동차일 가능성이 높다` 같은 판단에 유리합니다. 하지만 pooling을 많이 거쳤기 때문에 경계와 위치 정보는 거칠어집니다.

반대로 얕은 feature map은 의미 정보는 약하지만 edge, texture, 위치 정보가 더 잘 남아 있습니다.

FCN은 깊은 층의 score map만 사용하지 않고, 얕은 층의 feature map을 함께 사용하는 skip connection으로 더 정교한 예측을 만들 수 있음을 보여주었습니다.


In [ ]:
layers = ['shallow feature', 'middle feature', 'deep feature']
semantic_strength = [2, 5, 9]
location_detail = [9, 6, 2]

x = np.arange(len(layers))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, semantic_strength, width, label='semantic information')
plt.bar(x + width / 2, location_detail, width, label='location detail')
plt.xticks(x, layers)
plt.ylim(0, 10)
plt.title('깊이에 따른 정보의 성격')
plt.legend()
plt.show()


## 19-6. FCN의 전체 흐름

FCN을 한 줄로 정리하면 다음과 같습니다.

```text
image -> CNN feature extractor -> class score map -> upsampling -> pixel prediction
```

여기에 skip connection을 추가하면 다음처럼 볼 수 있습니다.

```text
deep semantic feature + shallow location feature -> refined segmentation map
```

이 관점은 다음에 배울 U-Net의 encoder-decoder 구조와도 자연스럽게 연결됩니다.


## 19-7. FCN이 남긴 관점

FCN의 의의는 특정 구현 하나에만 있지 않습니다. 더 중요한 점은 classification CNN을 dense prediction 문제에 맞게 재해석했다는 것입니다.

- CNN backbone은 이미지의 의미 feature를 추출할 수 있습니다.
- Fully connected layer를 convolution으로 바꾸면 공간 score map을 만들 수 있습니다.
- Downsampling으로 작아진 feature map은 upsampling으로 다시 키워야 합니다.
- 깊은 feature의 의미 정보와 얕은 feature의 위치 정보를 함께 쓰면 경계가 좋아질 수 있습니다.

이후 U-Net, DeepLab 계열, 여러 segmentation 모델은 이 기본 관점을 각자의 방식으로 발전시킵니다.


## 19-8. 정리

- FCN은 classification CNN을 segmentation에 맞게 바꾼 대표적인 초기 구조입니다.
- Fully connected layer를 convolution layer로 바꾸어 class score map을 만듭니다.
- Score map은 입력보다 작기 때문에 upsampling으로 원래 해상도에 맞춥니다.
- Skip connection은 깊은 층의 의미 정보와 얕은 층의 위치 정보를 결합합니다.
- 다음 노트북에서는 이 관점을 더 명확한 encoder-decoder 구조로 발전시킨 U-Net을 살펴봅니다.
